In [ ]:
# 모듈 불러오기

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# csv파일 읽어오기

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(train.shape)
print(test.shape)

train.head()

(1460, 81)
(1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [ ]:
# Id 분리

test_ids = test["Id"]

In [ ]:
# 이상치 정리

train = train.drop(train[(train["GrLivArea"]>4000)& (train["SalePrice"]<300000)].index)

print(train.shape)

(1458, 81)


In [ ]:
# 로그 변환

train["SalePrice"] = np.log1p(train["SalePrice"])

In [ ]:
#feature Engineering

# TotalSF
train["TotalSF"] = train["TotalBsmtSF"] + train["1stFlrSF"] + train["2ndFlrSF"]
test["TotalSF"] = test["TotalBsmtSF"] + test["1stFlrSF"] + test["2ndFlrSF"]

# TotalBathrooms
train["TotalBathrooms"] = (
    train["FullBath"]
    + train["HalfBath"] * 0.5
    + train["BsmtFullBath"]
    + train["BsmtHalfBath"] * 0.5
)

test["TotalBathrooms"] = (
    test["FullBath"]
    + test["HalfBath"] * 0.5
    + test["BsmtFullBath"]
    + test["BsmtHalfBath"] * 0.5
)

# HouseAge
train["HouseAge"] = train["YrSold"] - train["YearBuilt"]
test["HouseAge"] = test["YrSold"] - test["YearBuilt"]

# TotalRooms
train["TotalRooms"] = train["TotRmsAbvGrd"] + train["BedroomAbvGr"]
test["TotalRooms"] = test["TotRmsAbvGrd"] + test["BedroomAbvGr"]

In [7]:
# 결측치 처리

#LotFrontage
train["LotFrontage"] = train.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median())
)

test["LotFrontage"] = test.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median())
)

#None cols
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType"
]

for col in none_cols:
    if col in train.columns:
        train[col] = train[col].fillna("None")
    if col in test.columns:
        test[col] = test[col].fillna("None")

#Zerocols
zero_cols = [
    "MasVnrArea", "GarageYrBlt", "GarageArea", "GarageCars",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath"
]

for col in zero_cols:
    if col in train.columns:
        train[col] = train[col].fillna(0)
    if col in test.columns:
        test[col] = test[col].fillna(0)

#Mode & Median
# train
for col in train.select_dtypes(include=["object", "string"]).columns:
    train[col] = train[col].fillna(train[col].mode()[0])

for col in train.select_dtypes(include=["int64", "float64"]).columns:
    train[col] = train[col].fillna(train[col].median())

# test
for col in test.select_dtypes(include=["object", "string"]).columns:
    test[col] = test[col].fillna(test[col].mode()[0])

for col in test.select_dtypes(include=["int64", "float64"]).columns:
    test[col] = test[col].fillna(test[col].median())

print("train NaN 총개수:", train.isnull().sum().sum())
print("test NaN 총개수:", test.isnull().sum().sum())

train NaN 총개수: 0
test NaN 총개수: 0


In [8]:
#Id 제거

train = train.drop(["Id"],axis=1)
test = test.drop(["Id"],axis=1)

In [9]:
#One-Hot Encoding

train = pd.get_dummies(train)
test = pd.get_dummies(test)

In [10]:
#X,y 분리

y = train["SalePrice"]
X = train.drop("SalePrice",axis=1)

In [11]:
# train/test 컬럼 맞추기

X, test = X.align(test, join="left", axis= 1, fill_value=0)

print("X shape:", X.shape)
print("test shape:", test.shape)
print("SalePrice in X?", "SalePrice" in X.columns)
print("SalePrice in test?", "SalePrice" in test.columns)

X shape: (1458, 305)
test shape: (1459, 305)
SalePrice in X? False
SalePrice in test? False


In [12]:
#skew feature 찾기 + log 변환

skew = X.select_dtypes(include=["int64","float64"]).skew()
skew = skew[abs(skew)> 0.75]

print(skew.sort_values(ascending=False).head(10))

for col in skew.index:
    X[col] = np.log1p(X[col])
    test[col] = np.log1p(test[col])

MiscVal          24.460085
PoolArea         15.948945
LotArea          12.573925
3SsnPorch        10.297106
LowQualFinSF      9.004955
KitchenAbvGr      4.484883
BsmtFinSF2        4.251925
ScreenPorch       4.118929
BsmtHalfBath      4.100114
EnclosedPorch     3.087164
dtype: float64


In [13]:
# RandomForest CV

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

rf_scores = cross_val_score(
    rf,
    X,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

rf_rmse = -rf_scores
print("RandomForest RMSE:", rf_rmse)
print("평균:", rf_rmse.mean())

RandomForest RMSE: [0.13477814 0.13764379 0.14239898 0.13254199 0.13761663]
평균: 0.13699590511735982


In [ ]:
#Ridge CV

ridge = make_pipeline(
    StandardScaler(),
    Ridge(alpha=10)
)

ridge_scores = cross_val_score(
    ridge,
    X,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

ridge_rmse = -ridge_scores
print("Ridge RMSE:", ridge_rmse)
print("평균:", ridge_rmse.mean())

Ridge RMSE: [0.11721761 0.12763096 0.13735066 0.11092851 0.11988328]
평균: 0.12260220377078786


In [17]:
#Lasso CV

lasso = make_pipeline(
    StandardScaler(),
    Lasso(alpha=0.001, max_iter=10000)
)

lasso_scores = cross_val_score(
    lasso,
    X,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

lasso_rmse = -lasso_scores
print("Lasso RMSE:", lasso_rmse)
print("평균:", lasso_rmse.mean())

Lasso RMSE: [0.10762713 0.11815543 0.12669687 0.10823924 0.11410348]
평균: 0.11496442936147384


In [19]:
# LightGBM CV

lgbm = make_pipeline(
    StandardScaler(),
    LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=15,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
    )
)

lgbm_scores = cross_val_score(
    lgbm,
    X,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

lgbm_rmse = -lgbm_scores
print("LGBM RMSE:", lgbm_rmse)
print("평균:", lgbm_rmse.mean())

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001653 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3766
[LightGBM] [Info] Number of data points in the train set: 1166, number of used features: 199
[LightGBM] [Info] Start training from score 12.021352


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000956 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3772
[LightGBM] [Info] Number of data points in the train set: 1166, number of used features: 195
[LightGBM] [Info] Start training from score 12.023516


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001234 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3774
[LightGBM] [Info] Number of data points in the train set: 1166, number of used features: 199
[LightGBM] [Info] Start training from score 12.020683


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001353 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3789
[LightGBM] [Info] Number of data points in the train set: 1167, number of used features: 196
[LightGBM] [Info] Start training from score 12.032713


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001279 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3776
[LightGBM] [Info] Number of data points in the train set: 1167, number of used features: 198
[LightGBM] [Info] Start training from score 12.021807
LGBM RMSE: [0.12118919 0.13164444 0.13295182 0.11152547 0.12283613]
평균: 0.12402940969822848


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [21]:
ridge.fit(X,y)
lgbm.fit(X,y)
lasso.fit(X,y)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001426 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4058
[LightGBM] [Info] Number of data points in the train set: 1458, number of used features: 206
[LightGBM] [Info] Start training from score 12.024015


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('standardscaler', ...), ('lasso', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",0.001
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True


In [22]:
pred_lasso = lasso.predict(test)
pred_lgbm = lgbm.predict(test)

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [23]:
final_pred = (0.8 * pred_lasso) + (0.2 * pred_lgbm)

In [24]:
final_pred = np.expm1(final_pred)

In [25]:
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": final_pred
})

submission.head()

,Id,SalePrice
0,1461,121044.077341
1,1462,158016.984434
2,1463,186002.337001
3,1464,197398.705971
4,1465,197261.962267


In [26]:
submission.to_csv("sub_HP3.csv", index= False)

In [27]:
rf.fit(X,y)

importance = pd.Series(rf.feature_importances_, index= X.columns)
importance.sort_values(ascending=False).head(20)

OverallQual         0.446059
TotalSF             0.318538
GrLivArea           0.012164
GarageCars          0.011918
TotalBathrooms      0.010398
GarageArea          0.010363
LotArea             0.010351
BsmtFinSF1          0.010023
CentralAir_Y        0.009515
YearRemodAdd        0.009498
OverallCond         0.009459
CentralAir_N        0.009060
GarageYrBlt         0.007070
HouseAge            0.006996
BsmtUnfSF           0.006779
1stFlrSF            0.006381
YearBuilt           0.005630
LotFrontage         0.005140
TotalBsmtSF         0.004406
MSZoning_C (all)    0.003474
dtype: float64